# Retention & Session Analysis

**Question:** Does the GA4 ecommerce sample data show meaningful day-1 retention drop-off, and how does session activity trend over the period?

**Source models:** `fct_sessions_daily`, `fct_user_retention` (from the `my_project` dbt project).

This notebook pulls both marts, charts them, and records interpretation — the layer dbt itself doesn't give you.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import bigquery

PROJECT_ID = "your-gcp-project-id"
DATASET = "dbt_dev"

client = bigquery.Client(project=PROJECT_ID)

## Pull `fct_sessions_daily`

In [ ]:
query_sessions = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET}.fct_sessions_daily`
ORDER BY session_date
"""

df_sessions = client.query(query_sessions).to_dataframe()
df_sessions.head()

## Pull `fct_user_retention`

In [ ]:
query_retention = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET}.fct_user_retention`
ORDER BY first_session_date
"""

df_retention = client.query(query_retention).to_dataframe()
df_retention.head()

## Quick sanity check
Row counts and date ranges — confirm both marts pulled the full expected period before charting.

In [ ]:
print("sessions daily:", df_sessions.shape, df_sessions["session_date"].min(), "-", df_sessions["session_date"].max())
print("user retention:", df_retention.shape, df_retention["first_session_date"].min(), "-", df_retention["first_session_date"].max())

## Chart: sessions & purchase rate over time
*(Chart cells for the next session — data pull is the goal for today.)*

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(df_sessions["session_date"], df_sessions["total_sessions"], label="total_sessions", color="tab:blue")
ax1.set_ylabel("total_sessions")

ax2 = ax1.twinx()
ax2.plot(df_sessions["session_date"], df_sessions["purchase_rate"], label="purchase_rate", color="tab:orange")
ax2.set_ylabel("purchase_rate")

fig.suptitle("Daily sessions vs. purchase rate")
fig.autofmt_xdate()
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
plt.savefig("../assets/sessions_purchase_rate_chart.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart: day-1 retention by cohort date

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_retention["first_session_date"], df_retention["day_1_retention_rate"])
ax.set_ylabel("day_1_retention_rate")
ax.set_title("Day-1 retention by cohort date")
fig.autofmt_xdate()
plt.savefig("../assets/retention_chart.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df_sessions['session_date'] = pd.to_datetime(df_sessions['session_date'])
df_sessions['weekday'] = df_sessions['session_date'].dt.day_name()
df_sessions.groupby('weekday')['total_sessions'].mean().sort_values(ascending=False)

## Interpretation

- **Trend:** Total sessions and purchase rate move together — both show a clear
  spike around Jan 20-23, then decline toward the end of the month. Sessions
  also show a weekly pattern: Wednesday averages ~5000 sessions vs. Sunday's
  ~2,900, roughly a 40% swing between the highest and lowest days. Day-1
  retention hovers in the 3-6% range with no strong upward or downward trend
  across the period, aside from a dip in the last few observed cohorts.
  
- **Surprising finding:** Purchase rate is consistently low (~0.5-2%) even on
  peak-session days — sessions spiking doesn't mean conversion improves
  proportionally. The Jan 20-23 spike in particular looks like a traffic event
  (promo, campaign), since purchase_rate spikes with it.

- **Hypothesis to chase:** If I had channel/source data, I'd check whether the
  Jan 20-23 spike came from a specific acquisition channel — if it's paid
  traffic, the flat-to-low purchase rate might mean that channel is driving
  volume but not intent. Also worth checking whether day_1_retention_rate
  varies by cohort size — do the biggest cohorts retain worse (diluted by
  low-intent traffic) or the same as smaller ones?

- **Data quality note:** Initial total_sessions figures were undercounted by
  ~2% due to `count(distinct session_id)` collapsing sessions that share a
  session_id across different users (session_id is not globally unique in
  GA4). Caught by cross-checking against fct_funnel_purchase's independently
  calculated session count. Fixed in both fct_sessions_daily and
  fct_funnel_purchase.

In [ ]:
query_funnel = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET}.fct_funnel_purchase`
"""

df_funnel = client.query(query_funnel).to_dataframe()
df_funnel

In [ ]:
steps = ["total_sessions", "sessions_viewed_item", "sessions_added_to_cart", "sessions_purchased"]
values = df_funnel[steps].iloc[0]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(steps, values, color="tab:blue")
ax.set_ylabel("sessions")
ax.set_title("Funnel: sessions by step")

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, val, f"{val:,.0f}", ha="center", va="bottom")

plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("../assets/funnel_chart.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
drop_view = (values["total_sessions"] - values["sessions_viewed_item"]) / values["total_sessions"]
drop_cart = (values["sessions_viewed_item"] - values["sessions_added_to_cart"]) / values["sessions_viewed_item"]
drop_purchase = (values["sessions_added_to_cart"] - values["sessions_purchased"]) / values["sessions_added_to_cart"]

print(f"Session -> viewed item drop-off: {drop_view:.1%}")
print(f"Viewed item -> added to cart drop-off: {drop_cart:.1%}")
print(f"Added to cart -> purchased drop-off: {drop_purchase:.1%}")

## Interpretation: funnel drop-off

- **Finding:** Drop-off is remarkably consistent across all three funnel steps
  — 80.5% (session → view_item), 80.4% (view_item → add_to_cart), and 81.6%
  (add_to_cart → purchase). There isn't one dominant bottleneck; roughly 4 in
  5 sessions/users fail to progress at every single stage.

- **Why this matters:** A single dominant bottleneck (e.g. "90% never even
  view a product") would point to an acquisition/landing-page problem. A flat
  ~80% attrition rate at every step instead suggests either (a) a large
  proportion of low-intent traffic that behaves the same way regardless of
  stage, or (b) each step independently has friction worth investigating —
  slow page loads, unclear CTAs, checkout friction — rather than one fixable
  chokepoint.

- **Caveat:** This funnel counts event *presence* per session, not strict
  order (see model comment in fct_funnel_purchase.sql) — verified this
  affects only ~0.26% of sessions, so it doesn't explain the pattern.

- **Hypothesis to chase:** With channel/device/landing-page data, I'd check
  whether the ~80% non-progression rate is uniform across segments or
  concentrated in specific traffic sources — a uniform rate across segments
  would support the "broad low-intent traffic" theory over a UX-friction
  theory at any single step.